<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/NLPPytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer

In [9]:
#Encoding
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
print(tokenizer.vocab_size)

text = "This movie was great!"

input_ids = tokenizer(text, padding='max_length', max_length=200, truncation=True)

30522


In [10]:
max_words = tokenizer.vocab_size
max_len = 200

embed_dim = 256
num_heads = 4

ff_dim = 128
num_layers = 8
num_classes = 4

In [7]:
input_ids

{'input_ids': [101, 2023, 3185, 2001, 2307, 999, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [8]:
input_ids = input_ids['input_ids']

In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self,embed_dim,num_heads,ff_dim) -> None:
    super().__init__()

    #Brick2: multihead
    self.attention = nn.MultiheadAttention(embed_dim=embed_dim,num_heads=num_heads,dropout=0.2,batch_first=True)

    self.mlp = nn.Sequential(
        nn.Linear(in_features=embed_dim,out_features=ff_dim),
        nn.GELU(),
        nn.Linear(in_features=ff_dim,out_features=embed_dim),
        nn.Dropout(0.2)
    )

    self.layerNorm1 = nn.LayerNorm(embed_dim)
    self.layerNorm2 = nn.LayerNorm(embed_dim)

  def forward(self,input):
    att,_ = self.attention(input,input,input) # here it return new features and map of where they are pointing. for heatmap we use the second variable to check where the model is focusing on
    att = self.layerNorm1(input + att)

    mlp_Output = self.mlp(att)
    return self.layerNorm2(att + mlp_Output)

In [ ]:
class WordEmbedding(nn.Module):
  def __init__(self,max_words,embed_dim) -> None:
    super().__init__()
    self.wordEmbed = nn.Embedding(num_embeddings=max_words,embedding_dim=embed_dim)

  def forward(self,input):
    return self.wordEmbed(input)

In [ ]:
class NLPTransformer(nn.Module):
  def __init__(self,max_words,embed_dim,max_len,num_heads,ff_dim,num_layers,num_classes) -> None:
    super().__init__()
    self.wordEmbed = WordEmbedding(max_words=max_words,embed_dim=embed_dim)
    self.positionEmbed = nn.Parameter(torch.zeros(1,max_len,embed_dim))

    #transformer layer
    self.transformer_Layers = nn.ModuleList([
        TransformerBlock(embed_dim=embed_dim,num_heads=num_heads,ff_dim=ff_dim)
        for _ in range(num_layers)
    ])

    self.outputlayer = nn.Linear(embed_dim,num_classes)

  def forward(self,input):
    wordEmbedding = self.wordEmbed(input)
    x = wordEmbedding + self.positionEmbed

    for transformer_layer in self.transformer_Layers:
      x = transformer_layer(x)

    x = x.mean(dim=1)
    return self.outputlayer(x)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [ ]:

model = NLPTransformer(max_words=max_words,embed_dim=embed_dim,max_len=max_len,num_heads=num_heads,ff_dim=ff_dim,num_layers=num_layers,num_classes=num_classes).to(device)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimiser = optim.Adam(params=model.parameters(),lr= 0.0001)

In [ ]:
epochs = 10

for epoch in range(epochs):
  model.train()
  train_loss = 0
  train_correct = 0
  train_total = 0

  for input_ids,labels in train_loader:
    input_ids = input_ids.to(device)
    labels = labels.to(device)

    optimiser.zero_grad()

    output = model(input_ids)
    loss = loss_fn(output,labels)

    loss.backward()
    optimiser.step()

    train_loss += loss.item()
    _ , pred = torch.max(output,1)
    train_correct += (pred == labels).sum().item() # here pred has an array of label indexes for each images, and labels already have a real answers.
    train_total = labels.size(0)
  train_accuracy = train_correct / train_total
  train_losses = train_loss / len(train_loader)

  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0

  with torch.no_grad():
    for input_ids,labels in val_loader:
      input_ids = input_ids.to(device)
      labels = labels.to(device)

      output = model(input_ids)
      loss = loss_fn(output,labels)

      val_loss += loss.item()
      _ , pred = torch.max(output,1)
      val_correct += (pred == labels).sum().item() # here pred has an array of label indexes for each images, and labels already have a real answers.
      val_total = labels.size(0)
    val_accuracy = val_correct / val_total
    val_losses = val_loss / len(val_loader)

